In [8]:
import mlflow.sklearn
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
 
print("=" * 60)
print("RISK SCORING — All Patients at Discharge")
print("=" * 60)


StatementMeta(, d6eb5c01-6422-48d6-9a8b-e77e8514293a, 11, Finished, Available, Finished, False)

RISK SCORING — All Patients at Discharge


## Load Registered Model

In [9]:
# The model was registered in 04_ml_experiment.py.
# In production this would point to the 'Production' stage.
# During POC, use 'None' or specify the version number.
try:
    model = mlflow.sklearn.load_model(
        "models:/HospitalReadmission30d/Production"
    )
    print("Loaded model stage: Production")
except Exception:
    # During POC, model may not be promoted to Production yet
    model = mlflow.sklearn.load_model(
        "models:/HospitalReadmission30d/1"
    )
    print("Loaded model version: 1")

StatementMeta(, d6eb5c01-6422-48d6-9a8b-e77e8514293a, 12, Finished, Available, Finished, False)

/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/mlflow/store/artifact/utils/models.py:32: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.12.2/model-registry.html#migrating-from-stages
  latest = client.get_latest_versions(name, None if stage is None else [stage])


Loaded model version: 1


StatementMeta(, d6eb5c01-6422-48d6-9a8b-e77e8514293a, 13, Finished, Available, Finished, False)

##  Load Features

In [10]:
FEATURE_COLS = [
    "time_in_hospital","num_lab_procedures","num_procedures",
    "num_medications","number_diagnoses",
    "number_inpatient","number_emergency","number_outpatient",
    "prior_visits_total","is_high_prior_use",
    "has_prior_inpatient","has_prior_emergency",
    "is_long_stay","is_polypharmacy","is_complex_patient",
    "age_ord",
    "total_med_changes","total_meds_taken",
    "insulin_changed","insulin_increased",
    "A1C_tested","A1C_high","A1C_normal","glucose_tested",
    "is_diabetes_primary","has_circulatory_dx",
    "diag1_cat_idx","diag2_cat_idx","diag3_cat_idx",
    "gender_idx","race_idx",
]
 
df_spark  = spark.read.format("delta").table("silver_features")
available = [f for f in FEATURE_COLS if f in df_spark.columns]
 
df_pd = df_spark.select(
    ["encounter_id","patient_nbr","readmitted_30d"] + available
).toPandas()
print(f"Patients to score: {len(df_pd):,}")

StatementMeta(, d6eb5c01-6422-48d6-9a8b-e77e8514293a, 14, Finished, Available, Finished, False)

Patients to score: 101,766


## Score Every Patient

In [11]:
X_score = df_pd[available].fillna(0)
df_pd["risk_score_pct"] = (
    model.predict_proba(X_score)[:, 1] * 100
).round(1)

StatementMeta(, d6eb5c01-6422-48d6-9a8b-e77e8514293a, 15, Finished, Available, Finished, False)

## Assign Risk Tiers

In [13]:
# Tier thresholds are the key clinical decision:
#   HIGH   (≥35%): care coordinator call before discharge
#   MEDIUM (20–35%): schedule GP follow-up appointment
#   LOW    (<20%): standard discharge process
#
# 35% threshold was chosen to balance:
#   - Sensitivity (catching most readmissions)
#   - Burden (how many patients care team must review)
# For a live client, calibrate against their actual capacity.
 
# df_pd["risk_tier"] = pd.cut(
#     df_pd["risk_score_pct"],
#     bins=[-1, 20, 35, 101],
#     labels=["LOW", "MEDIUM", "HIGH"]
# ).astype(str)
 
df_pd["actual_readmit_30d"] = df_pd["readmitted_30d"]

StatementMeta(, d6eb5c01-6422-48d6-9a8b-e77e8514293a, 17, Finished, Available, Finished, False)

In [17]:
# The model's raw scores cluster between 14–85 with median ~47.
# A fixed 35% threshold flags 80% as HIGH — not actionable.
# Instead, derive thresholds from the score distribution so
# the HIGH tier contains a manageable ~20% of patients.

# Replace the pd.cut block in notebook 05 with this:

scores = df_pd["risk_score_pct"]
p80 = scores.quantile(0.80)
p50 = scores.quantile(0.50)

print(f"Thresholds → HIGH: ≥{p80:.1f}%  |  MEDIUM: {p50:.1f}–{p80:.1f}%  |  LOW: <{p50:.1f}%")

df_pd["risk_tier"] = pd.cut(
    scores, bins=[-1, p50, p80, 101],
    labels=["LOW","MEDIUM","HIGH"]
).astype(str)

# Then run the sensitivity table to pick the right threshold
total_pos = df_pd["actual_readmit_30d"].sum()
print(f"\n{'Threshold':>10} {'HIGH pts':>10} {'% all':>7} {'Captured':>10} {'Capture%':>10} {'Precision':>10}")
for t in [45, 50, 55, 60, 65, 70]:
    mask  = df_pd["risk_score_pct"] >= t
    pts   = mask.sum()
    capt  = df_pd.loc[mask, "actual_readmit_30d"].sum()
    print(f"  ≥{t}%     {pts:>9,}  {pts/len(df_pd)*100:>6.1f}%  {capt:>9,}  "
          f"{capt/total_pos*100:>8.1f}%  {capt/pts*100:>8.1f}%")

StatementMeta(, d6eb5c01-6422-48d6-9a8b-e77e8514293a, 21, Finished, Available, Finished, False)

Thresholds → HIGH: ≥56.2%  |  MEDIUM: 46.7–56.2%  |  LOW: <46.7%

 Threshold   HIGH pts   % all   Captured   Capture%  Precision
  ≥45%        56,632    55.6%      8,196      72.2%      14.5%
  ≥50%        39,898    39.2%      6,488      57.1%      16.3%
  ≥55%        24,233    23.8%      4,598      40.5%      19.0%
  ≥60%        12,094    11.9%      2,760      24.3%      22.8%
  ≥65%         6,015     5.9%      1,657      14.6%      27.5%
  ≥70%         2,349     2.3%        828       7.3%      35.2%


## Threshold Sensitivity Table

In [14]:
# Show capture rate vs burden across multiple HIGH thresholds.
# This is the table you show the clinical manager to decide
# what workload their care coordinators can absorb.

print("\nThreshold sensitivity analysis:")
print(f"{'HIGH threshold':>16} {'HIGH patients':>14} {'% of all':>9} "
      f"{'Readmits caught':>16} {'Capture rate':>13} {'Precision':>10}")
print("-" * 82)

total_pos = df_pd["actual_readmit_30d"].sum()
total_pat = len(df_pd)

for threshold in [40, 45, 50, 55, 60, 65, 70]:
    high_mask   = df_pd["risk_score_pct"] >= threshold
    high_pts    = high_mask.sum()
    high_pct    = high_pts / total_pat * 100
    caught      = df_pd.loc[high_mask, "actual_readmit_30d"].sum()
    capture     = caught / total_pos * 100
    precision   = caught / high_pts * 100 if high_pts > 0 else 0
    print(f"  ≥ {threshold}%          {high_pts:>10,}     {high_pct:>6.1f}%"
          f"       {caught:>9,}          {capture:>7.1f}%    {precision:>7.1f}%")

StatementMeta(, d6eb5c01-6422-48d6-9a8b-e77e8514293a, 18, Finished, Available, Finished, False)


Threshold sensitivity analysis:
  HIGH threshold  HIGH patients  % of all  Readmits caught  Capture rate  Precision
----------------------------------------------------------------------------------
  ≥ 40%              71,242       70.0%           9,441             83.1%       13.3%
  ≥ 45%              56,632       55.6%           8,196             72.2%       14.5%
  ≥ 50%              39,898       39.2%           6,488             57.1%       16.3%
  ≥ 55%              24,233       23.8%           4,598             40.5%       19.0%
  ≥ 60%              12,094       11.9%           2,760             24.3%       22.8%
  ≥ 65%               6,015        5.9%           1,657             14.6%       27.5%
  ≥ 70%               2,349        2.3%             828              7.3%       35.2%


## Performance Summary

In [22]:
print("\nRisk tier distribution:")
tier_counts = df_pd["risk_tier"].value_counts()
for tier, cnt in tier_counts.items():
    pct = cnt / len(df_pd) * 100
    print(f"  {tier:<8}: {cnt:>6,} patients ({pct:.1f}%)")
 
# Capture rate: what % of actual readmissions does HIGH tier catch?
total_pos = df_pd["actual_readmit_30d"].sum()
high_df   = df_pd[df_pd["risk_tier"] == "HIGH"]
caught    = high_df["actual_readmit_30d"].sum()
precision = caught / len(high_df) if len(high_df) > 0 else 0
 
print(f"\nModel capture analysis (threshold = %):")
print(f"  Total 30-day readmissions : {total_pos:,}")
print(f"  Caught by HIGH tier       : {caught:,} ({caught/total_pos:.0%})")
print(f"  HIGH tier precision       : {precision:.1%}")
print(f"  False positives ratio     : 1 readmit per {len(high_df)//max(caught,1):.0f} HIGH-flagged patients")

StatementMeta(, d6eb5c01-6422-48d6-9a8b-e77e8514293a, 26, Finished, Available, Finished, False)


Risk tier distribution:
  LOW     : 51,154 patients (50.3%)
  MEDIUM  : 30,455 patients (29.9%)
  HIGH    : 20,157 patients (19.8%)

Model capture analysis (threshold = %):
  Total 30-day readmissions : 11,357
  Caught by HIGH tier       : 4,052 (36%)
  HIGH tier precision       : 20.1%
  False positives ratio     : 1 readmit per 4 HIGH-flagged patients


##  Write Risk Scores

In [19]:
df_out = df_pd[[
    "encounter_id", "patient_nbr",
    "risk_score_pct", "risk_tier",
    "actual_readmit_30d"
]]
df_spark_out = spark.createDataFrame(df_out)
df_spark_out.write.format("delta").mode("overwrite") \
            .option("overwriteSchema","true") \
            .saveAsTable("silver_risk_scores")
 
print(f"\n silver_risk_scores: {df_spark_out.count():,} rows")
print("\nRisk score distribution:")
df_spark_out.select(
    F.min("risk_score_pct").alias("min"),
    F.avg("risk_score_pct").alias("mean"),
    F.percentile_approx("risk_score_pct", 0.5).alias("median"),
    F.percentile_approx("risk_score_pct", 0.75).alias("p75"),
    F.max("risk_score_pct").alias("max")
).show()
print("Proceed to → 06_gold_layer.sql in Fabric Warehouse")

StatementMeta(, d6eb5c01-6422-48d6-9a8b-e77e8514293a, 23, Finished, Available, Finished, False)


 silver_risk_scores: 101,766 rows

Risk score distribution:
+----+-----------------+------+----+----+
| min|             mean|median| p75| max|
+----+-----------------+------+----+----+
|14.1|46.11909380373878|  46.7|54.6|85.0|
+----+-----------------+------+----+----+

Proceed to → 06_gold_layer.sql in Fabric Warehouse


In [20]:
display(df_spark_out)

StatementMeta(, d6eb5c01-6422-48d6-9a8b-e77e8514293a, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 10033ab8-84c6-4968-8874-e7d6d3709f25)

In [21]:
# verify
df_verify = spark.read.format("delta").table("silver_risk_scores")

total = df_verify.count()
print(f"Total rows: {total:,}")          # 101,766

# Tier distribution
df_verify.groupBy("risk_tier").count() \
         .orderBy("count", ascending=False).show()
# LOW ~50k, MEDIUM ~30k, HIGH ~20k

# Capture rate
from pyspark.sql import functions as F
total_pos = df_verify.agg(F.sum("actual_readmit_30d")).collect()[0][0]
high_caught = df_verify.filter(F.col("risk_tier") == "HIGH") \
                       .agg(F.sum("actual_readmit_30d")).collect()[0][0]
print(f"\nTotal actual readmissions : {total_pos:,}")
print(f"Caught by HIGH tier       : {high_caught:,} ({high_caught/total_pos:.0%})")
print(f"HIGH tier precision       : {high_caught/df_verify.filter(F.col('risk_tier')=='HIGH').count():.1%}")

# Schema
df_verify.printSchema()

# Score distribution
df_verify.select(
    F.min("risk_score_pct").alias("min"),
    F.avg("risk_score_pct").alias("mean"),
    F.percentile_approx("risk_score_pct", 0.5).alias("p50"),
    F.percentile_approx("risk_score_pct", 0.8).alias("p80"),
    F.max("risk_score_pct").alias("max")
).show()

StatementMeta(, d6eb5c01-6422-48d6-9a8b-e77e8514293a, 25, Finished, Available, Finished, False)

Total rows: 101,766
+---------+-----+
|risk_tier|count|
+---------+-----+
|      LOW|51154|
|   MEDIUM|30455|
|     HIGH|20157|
+---------+-----+


Total actual readmissions : 11,357
Caught by HIGH tier       : 4,052 (36%)
HIGH tier precision       : 20.1%
root
 |-- encounter_id: integer (nullable = true)
 |-- patient_nbr: integer (nullable = true)
 |-- risk_score_pct: float (nullable = true)
 |-- risk_tier: string (nullable = true)
 |-- actual_readmit_30d: integer (nullable = true)

+----+-----------------+----+----+----+
| min|             mean| p50| p80| max|
+----+-----------------+----+----+----+
|14.1|46.11909380373878|46.7|56.2|85.0|
+----+-----------------+----+----+----+



In [ ]:
# Notebook: 05_risk_scoring.py
# Scenario 02: Healthcare Patient Readmission Prediction
# Input:  silver_features + registered MLflow model
# Output: silver_risk_scores (risk_score_pct, risk_tier)